In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Numerical Stability and Conditioning

Every computation in this chapter so far ran in floating point, and we read
its results as if they were exact real numbers. Most of the time, that
convenient fiction holds up. Occasionally, however, a training loss turns into
`NaN`, or a linear solver returns an answer that looks completely plausible
and is badly wrong. In this section we ask when finite-precision arithmetic,
rather than the optimization algorithm, decides the answer—and what we can do
about it.

Two small examples show the two ways numerical trouble arrives. First, the
three logits $(1000, 1001, 1002)$ define an entirely ordinary softmax, with
probabilities $(0.09, 0.24, 0.67)$. Yet if we evaluate the textbook formula as
written, it returns `NaN` three times, even in double precision (fp64).
Second, `np.linalg.solve` applied to a $12\times12$ linear system returns an
answer that is off by 30%, even though it is the exact solution of a system
that differs from the stated one only in the seventeenth decimal digit. The
first failure halts training with a `NaN` loss; the second slips past
unnoticed.

These two failures have fundamentally different causes. The naive softmax
formula passes through $e^{1000}$, a number too large for any standard
floating-point format to hold, whereas an algebraically equivalent rewrite
that never forms $e^{1000}$ is accurate to a few units in the last place. By
contrast, the $12\times12$ linear system is so sensitive that a rounding error
in the sixteenth digit of its data moves the solution in the first digit, and
no rewriting of the solver can undo that sensitivity.

To tell those two situations apart, we begin by examining how computers store
real numbers and how much error each rounding introduces. Next, we rewrite
softmax and cross-entropy so they never pass through numbers the format cannot
hold, and we rewrite the variance so it never subtracts nearly equal large
numbers. We then introduce a sensitivity measure—the condition number of a
function—that separates a bad algorithm from an inherently hard problem.
Finally, we see how the condition number of a linear system governs both the
digits a least-squares solve can keep and, as in
that section, the number of steps gradient
descent requires. While that section shows how modern training uses
low-precision formats in practice, here we derive why computations in them
succeed or fail.

Three ideas matter more than any individual formula: the relative rounding
model for floating-point arithmetic, the principle that an unstable algorithm
routes a well-conditioned problem through an ill-conditioned intermediate
step, and the bound linking a solver's solution error to the problem's
sensitivity. The subsections on summation order and ridge regularization can
be skipped on a first reading.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import ml_dtypes
from fractions import Fraction

## Floating-Point Arithmetic

### Representation and Spacing

What does a computer actually store when we ask it to store $1.1$? The
mechanism is easiest to see in a toy binary format that keeps only two bits
after the binary point. Between $1$ and $2$, this format can hold just four
values: $(1.00)_2=1$, $(1.01)_2=1.25$, $(1.10)_2=1.5$, and $(1.11)_2=1.75$.
Multiplying these four values by powers of two fills out the other intervals:
$2, 2.5, 3, 3.5$ between $2$ and $4$, and $0.5, 0.625, 0.75, 0.875$ between
$0.5$ and $1$. Each *octave*—an interval $[2^e, 2^{e+1})$ between consecutive
powers of two—holds exactly four values, so the absolute gap between adjacent
numbers doubles every time the magnitude doubles.

Any real number that falls between grid points is rounded to the nearest one:
$1.1\mapsto1$ (a relative error of $-0.091$), $1.3\mapsto1.25$ (an error of
$-0.038$), and $3.3\mapsto3.5$ (an error of $+0.061$). Halfway points are
ties, and standard arithmetic breaks ties toward the neighbor whose last bit
is $0$: the midpoint $1.125$ therefore rounds to $1$, an error of $-0.111$.
Notice that no relative error exceeds $1/8$. Rounding moves a number by at
most half a gap, and in every octave half a gap is $1/8$ of the octave's
smallest value. the figure sketches this grid on the
real line.

![Floating-point numbers on the real line for a toy format with two mantissa bits (four values per octave, $\varepsilon_{\mathrm{mach}}=1/4$, smallest normal $0.25$). The absolute gap between neighbors doubles at every power of two while the relative gap stays near $\varepsilon_{\mathrm{mach}}$. Below the smallest normal, subnormal numbers continue with the same spacing down to zero; after the axis break, real formats end at their overflow thresholds (fp16 at $65504$, fp32 at about $3.4\times10^{38}$).](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-fp-number-line.svg)

Real floating-point formats work in the same way, simply with more bits. A
floating-point number is scientific notation in base two with a fixed budget
of $p$ binary digits after the binary point,

$$
x = (-1)^s \cdot (1.b_1 b_2 \ldots b_p)_2 \cdot 2^{e},
$$

specified by a sign bit $s$, *mantissa* (or significand) bits
$b_1,\ldots,b_p$, and an integer exponent $e$ drawn from a fixed range. The
exponent sets the range of magnitudes the format can represent, while the
mantissa sets its relative precision. Within the octave $[2^e, 2^{e+1})$,
adjacent values are spaced $2^{e-p}$ apart. The gap from $1$ to the next
representable number, $\varepsilon_{\mathrm{mach}}=2^{-p}$, is called
*machine epsilon*; our toy format has $p=2$ and
$\varepsilon_{\mathrm{mach}}=1/4$.

Rounding a real number $x$ to the nearest representable value moves it by at
most half a gap, $2^{e-p-1}$. Because $|x|\ge2^e$ throughout the octave, the
relative error is at most $2^{-p-1}$. Calling a value *normal* when it has the
form the equation with an in-range exponent, every real
$x$ that rounds to a normal number satisfies

$$
\mathrm{fl}(x) = x\,(1 + \delta), \qquad |\delta| \le u = \tfrac12\,\varepsilon_{\mathrm{mach}} = 2^{-p-1} .
$$

In words, each rounding perturbs a number by at most one part in $2^{p+1}$:
one part in $8$ for our toy format, about one part in 17 million for fp32
($p=23$), and one part in $256$ for bf16 ($p=7$). The bound $u$ is called the
*unit roundoff*. Standard IEEE arithmetic rounds the result of every basic
operation to the nearest representable value, so the computed sum
$x\oplus y$ equals $(x+y)(1+\delta)$ with $|\delta|\le u$, and the same
guarantee holds for $-$, $\times$, and $/$
[@IEEE.754.2019; @Goldberg.1991].

This clean relative model has three exceptions. Results larger than the format
can hold *overflow* to $\infty$. Results smaller than the smallest normal
number *underflow*, stepping down through *subnormal* numbers that fill the
remaining gap to zero. Undefined operations, such as $0/0$ or $\infty/\infty$,
produce `NaN`. All three are absolute-scale effects rather than relative ones,
and we will return to them right after putting the standard formats on the
table.

### Formats

Deep learning relies on five formats narrower than double precision (fp64).
Let us inspect each format's machine epsilon $\varepsilon_{\mathrm{mach}}$,
smallest normal number, and largest finite value directly from the library:

In [ ]:
print(f'{"dtype":>14} {"eps":>10} {"smallest normal":>16} {"max":>10}')
for dt in [np.float16, np.float32, ml_dtypes.bfloat16,
           ml_dtypes.float8_e4m3fn, ml_dtypes.float8_e5m2]:
    fi = ml_dtypes.finfo(dt)                   # values have dtype dt
    print(f'{np.dtype(dt).name:>14} {float(fi.eps):10.3e} '
          f'{float(fi.smallest_normal):16.3e} {float(fi.max):10.3e}')

**fp32** keeps $p=23$ mantissa bits—about seven decimal digits—and reaches up
to $3.4\times10^{38}$. **fp16** keeps $p=10$ bits (about three decimal digits)
but has a narrow exponent range: it overflows above $65504$, and below
$2^{-14}\approx6.1\times10^{-5}$ its values become subnormal. **bf16**
(`bfloat16` in the libraries) makes the opposite trade-off. It keeps all eight
exponent bits of fp32, preserving its vast dynamic range, at the cost of
keeping only $p=7$ mantissa bits ($\varepsilon_{\mathrm{mach}}=2^{-7}\approx
0.0078$, roughly two to three decimal digits). Notice that the value $2^{-8}$
often quoted for bf16 is its unit roundoff $u=\varepsilon_{\mathrm{mach}}/2$.
The two fp8 formats push this trade-off further still: E4M3 keeps three
mantissa bits (about one decimal digit) and tops out at $448$, while E5M2
keeps two mantissa bits with fp16's exponent range. With so little dynamic
range, fp8 is practical only when paired with a separate scale factor per
tensor or per block of values.

Because bf16 is simply the top 16 bits of an fp32 number—the same sign bit,
the same eight exponent bits, and the first seven mantissa bits—converting
fp32 to bf16 amounts to rounding off the low 16 bits. We can implement that
rounding directly with integer arithmetic on the raw bit pattern. The midpoint
of the discarded 16 bits is `0x8000`, so we add `0x7FFF` (one below the
midpoint), plus one more if the lowest kept bit is $1$, and then clear the
low 16 bits. This sum carries into the kept bits if and only if the discarded
part lies strictly above the midpoint, or sits right on the midpoint with an
odd kept bit. That is round-to-nearest, ties-to-even in a single line of bit
manipulation; Exercise
5 builds
stochastic rounding on top of it. Let us verify it on one input above a
midpoint and two exact ties:

In [ ]:
def to_bf16(x):
    """Round float32 to the nearest bfloat16, ties to even."""
    bits = np.atleast_1d(np.asarray(x, np.float32)).view(np.uint32)
    bits = (bits + 0x7FFF + ((bits >> 16) & 1)) & 0xFFFF0000
    return bits.astype(np.uint32).view(np.float32)

for x, expected in [(1 + 2**-8 + 2**-10, 1 + 2**-7),   # above the midpoint
                    (1 + 2**-8, 1.0),                  # tie, 1 is even
                    (1 + 3 * 2**-8, 1 + 2**-6)]:       # tie, 1 + 2^-6 is even
    print(f'{x:.8f} -> {to_bf16(x).item():.8f}   (expected {expected:.8f})')

As expected, the first input lies above the midpoint between $1$ and
$1+2^{-7}$ and rounds up. The second sits right on that midpoint and rounds
down to the even neighbor $1$, whereas the third sits on the midpoint between
$1+2^{-7}$ and $1+2^{-6}$ and rounds up to the even neighbor $1+2^{-6}$. Next,
let us check the same tie rule in fp32, observe how the spacing grows with
magnitude, and find the largest logit each format can exponentiate without
overflow:

In [ ]:
eps = np.finfo(np.float32).eps
one = np.float32(1.0)
print('1 + eps   != 1 :', one + eps != one)
print('1 + eps/2 == 1 :', one + np.float32(eps / 2) == one)
print('gap between float32 neighbors at 1    :', np.spacing(one))
print('gap between float32 neighbors at 2^20 :', np.spacing(np.float32(2.0**20)))
for dt in [np.float16, np.float32, np.float64]:
    print(f'{np.dtype(dt).name}: exp(x) overflows for x >',
          f'{np.log(np.finfo(dt).max):.2f}')

Here $1+\varepsilon_{\mathrm{mach}}/2$ is the midpoint between $1$ and its
upper neighbor, so ties-to-even sends it back to $1$. The worst-case rounding
error of half a gap really does occur, giving a relative error of $u/(1+u)$,
just inside the bound $|\delta|\le u$. Meanwhile, the gap at $2^{20}$ is
$2^{20}$ times—about a million times—the gap at $1$. Finally, the last three
lines pinpoint where $e^x$ overflows: above $11.09$ in fp16, $88.72$ in fp32,
and $709.78$ in fp64.

### Overflow, Underflow, and Small Updates

Because the exponential turns an additive scale into a multiplicative one,
even modest logits can overflow. In fp16, the ceiling
$\log 65504\approx11.09$ sits well inside the range of ordinary unnormalized
classifier scores.

At the opposite end, $e^{-x}$ underflows. Below the smallest normal number, a
format steps through subnormal numbers that keep the same absolute spacing
while shedding significant bits one by one, until values round all the way to
$0$. Taking $\log 0$ later produces $-\infty$, and the backward pass fills
with `inf` or `NaN`. In fp16, subnormals begin below
$2^{-14}\approx6.1\times10^{-5}$, the smallest subnormal is
$2^{-24}\approx6\times10^{-8}$, and anything below
$2^{-25}\approx3\times10^{-8}$ rounds straight to zero.

Addition introduces a second absolute threshold that matters directly for
training. A gradient step replaces a weight $w$ by $w-\eta g$. If the update
$\eta g$ is small compared with the spacing around $w$, the sum simply rounds
back to $w$, and the step vanishes without a trace.

**Proposition (small updates vanish).** *Let $w>0$ be a floating-point number
with $2^e\le w<2^{e+1}$ in a format with $p$ mantissa bits, and let
$\Delta$ be real. If $|\Delta|<2^{e-p-1}$, then $\mathrm{fl}(w+\Delta)=w$,
except when $w=2^e$ and $\Delta<0$; in that case the same holds for
$|\Delta|<2^{e-p-2}$.*

**Proof.** In $[2^e, 2^{e+1}]$ the representable numbers are $2^{e-p}$ apart,
so the neighbors of $w$ are $w\pm2^{e-p}$, except that below $w=2^e$ the
lower octave $[2^{e-1},2^e)$ begins and the spacing halves to $2^{e-p-1}$.
Rounding to the nearest number returns $w$ whenever $w+\Delta$ is closer to
$w$ than to the adjacent grid point on its side—that is, whenever $|\Delta|$
is strictly less than half of that gap. $\blacksquare$

For a weight update $\Delta=-\eta g$ with $2^e<w<2^{e+1}$, the proposition
reads

$$
\mathrm{fl}(w-\eta g)=w \qquad \textrm{whenever } |\eta g|<2^{e-p-1}=u\,2^e .
$$

In short, any update smaller than roughly $u|w|$ is lost. The relative model
the equation describes products and quotients well, but
when we add two numbers of very different magnitudes, it hides this sharp
absolute cutoff.

A weight sitting right on a power of two, such as $w=1$, falls into the
exceptional case of the proposition: because the spacing halves below $1$, an
increase vanishes below $u$ (as that section tests), whereas a
decrease already vanishes below $u/2$. In bf16, a weight equal to $1$ therefore
ignores any decrease smaller than $2^{-9}\approx0.002$, so a learning rate of
$10^{-3}$ with an order-$1$ gradient leaves the weight completely frozen. The
corresponding threshold is $2^{-12}\approx2.4\times10^{-4}$ in fp16 and
$2^{-25}\approx3\times10^{-8}$ in fp32—which is why mixed-precision training
keeps a master copy of the weights in fp32. Let us test steps straddling these
thresholds (computing the bf16 updates in fp32 and rounding with `to_bf16`):

In [ ]:
one32, one16 = np.float32(1.0), np.float16(1.0)
print('bfloat16: 1 - 1e-3 ->', to_bf16(one32 - np.float32(1e-3)).item())
print('bfloat16: 1 - 2e-3 ->', to_bf16(one32 - np.float32(2e-3)).item())
print('fp16    : 1 - 1e-4 ->', one16 - np.float16(1e-4))
print('fp16    : 1 - 3e-4 ->', one16 - np.float16(3e-4))
print('fp32    : 1 - 1e-3 ->', one32 - np.float32(1e-3))

Sure enough, a step of $10^{-3}$ leaves the bf16 weight stuck at $1$, whereas
$2\times10^{-3}$ clears the $0.00195$ threshold and jumps to the neighbor
$1-2^{-8}=0.99609375$. Similarly, fp16 swallows $10^{-4}$ and responds to
$3\times10^{-4}$ by stepping to $1-2^{-11}\approx0.9995$, while fp32 keeps the
full update.

Gradients run into the underflow threshold instead. In fp16, gradients below
$6.1\times10^{-5}$ lose precision and those below $3\times10^{-8}$ vanish to
zero. Multiplying the loss by a constant factor $S$ before the backward pass
scales every gradient up by $S$ and lifts them back into the normal range;
dividing by $S$ afterward in fp32 restores the true scale
(that section covers the practical workflow, and Exercise
1 determines
the admissible window for $S$).

*Stochastic rounding* [@Gupta.Agrawal.Gopalakrishnan.ea.2015] removes the
lost-update cutoff in a different way. Instead of always rounding to the
nearest grid point, it rounds up or down at random with probabilities chosen
so that the expected stored value equals the exact result. Updates far too
small for round-to-nearest then still accumulate over many steps in
expectation
(Exercise 5).

## Stable Softmax and Cross-Entropy

Softmax, log-sum-exp, and cross-entropy all exponentiate logits, so all three
collide head-on with the overflow and underflow thresholds we just computed.
To keep everything concrete, we will carry one running example through this
subsection: a two-class classifier with logits $\mathbf z=(0,t)$ whose true
label is the unlikely class $0$. Its probabilities are
$\mathrm{softmax}(\mathbf z)=\bigl(1/(1+e^t),\, e^t/(1+e^t)\bigr)$, its
log-sum-exp is $\log(1+e^t)$, and the cross-entropy loss for class $0$ is
$\log(1+e^t)\approx t$ for large $t$. None of these outputs is extreme: even
at $t=100$, the probabilities are $3.7\times10^{-44}$ and $1$, and the loss
is simply $100$.

### The Max Shift

Given logits $\mathbf z\in\mathbb R^n$, the softmax vector is defined by

$$
\mathrm{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^n e^{z_j}} .
$$

At $t=100$, evaluating this formula directly forms $e^{100}$, which exceeds
fp32's overflow ceiling of $88.72$. The second entry turns into `inf/inf`
(`NaN`), and the first collapses to $1/\infty=0$. Notice that the true
probabilities themselves fit easily in fp32; only the intermediate exponential
does not.

Luckily, multiplying numerator and denominator by the same positive constant
$e^{-c}$ leaves the ratio untouched, so softmax is *shift-invariant*:

$$
\mathrm{softmax}(\mathbf{z} - c\mathbf{1}) = \mathrm{softmax}(\mathbf{z})
\qquad \textrm{for every } c \in \mathbb{R}.
$$

Factoring $e^c$ out of the sum inside a logarithm gives the companion
identity for **log-sum-exp**, $\mathrm{lse}(\mathbf{z}) = \log \sum_j e^{z_j}$,
which is the logarithm of the softmax denominator:

$$
\mathrm{lse}(\mathbf{z}) = c + \log \sum_{j=1}^n e^{z_j - c} .
$$

Both identities hold for any $c\in\mathbb{R}$, and choosing $c=\max_j z_j$
makes both computations safe. Every shifted exponent $z_j-c$ is then at most
$0$, so no exponential can overflow. At the same time, the largest term
equals $e^0=1$, so the sum is guaranteed to lie in $[1,n]$ and its logarithm
cannot underflow to $-\infty$. Let us test this on $(0, 100)$ in fp32, on the
opening's $(1000, 1001, 1002)$ in fp64, and on $(0, 10)$ where both versions
stay finite—do they return the exact same bits there?

In [ ]:
def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum()

def softmax_stable(z):
    e = np.exp(z - z.max())        # shift by the max: largest exponent is 0
    return e / e.sum()

z = np.array([0.0, 100.0], dtype=np.float32)
with np.errstate(over='ignore', invalid='ignore'):
    print('naive  softmax(0, 100):', softmax_naive(z))
print('stable softmax(0, 100):', softmax_stable(z))
z = np.array([1000.0, 1001.0, 1002.0])        # the opening's logits, fp64
with np.errstate(over='ignore', invalid='ignore'):
    print('naive  softmax(1000, 1001, 1002):', softmax_naive(z))
print('stable softmax(1000, 1001, 1002):', softmax_stable(z).round(4))
z = np.array([0.0, 10.0], dtype=np.float32)
a, b = softmax_naive(z), softmax_stable(z)
print('softmax(0, 10), naive minus stable, in ulps:',
      (a - b) / np.spacing(b))

At $t=100$, the shifted softmax returns $(3.8\times10^{-44}, 1)$—the true
$3.7\times10^{-44}$ rounded onto the subnormal grid with five significant
bits left—while the direct formula returns $0$ and `NaN`. Switching to higher
precision does not cure the problem: the logits $(1000, 1001, 1002)$ from the
opening overflow $e^{1000}$ even in fp64, turning the direct formula into
three `NaN`s where the shifted version cleanly returns $(0.09, 0.24, 0.67)$.
At $t=10$, the two formulas either agree to the last bit or differ by one or
two units in the last place (*ulps*), because real-number identities do not
guarantee identical rounding orders. One edge case remains: the max shift
cannot rescue non-finite logits—if an entire row is $-\infty$, then
$c=-\infty$ and $z_j-c$ is undefined.

### Bounds for Log-Sum-Exp

Log-sum-exp is the log-normalizer of every exponential-family model
(that section), and up to a single subtraction it is
the cross-entropy loss. The shift the equation immediately
tells us how close it sits to the largest logit:

**Proposition (log-sum-exp sandwich).** *For every
$\mathbf{z} \in \mathbb{R}^n$,*

$$
\max_j z_j \;\le\; \mathrm{lse}(\mathbf{z}) \;\le\; \max_j z_j + \log n .
$$

**Proof.** Set $c = \max_j z_j$ in the equation. Every
exponential $e^{z_j - c}$ lies in $(0, 1]$ and the maximizing term equals
$1$, so the sum lies in $[1, n]$ and its logarithm in $[0, \log n]$. Adding
$c$ yields both bounds. $\blacksquare$

Log-sum-exp is therefore a *soft maximum*. In our two-class example,
$\mathrm{lse}(0,t)=\log(1+e^t)$ stays within $\log 2\approx0.69$ of
$\max(0,t)$, attaining the full gap at $t=0$ where the two logits tie and
hugging the maximum closely as $|t|$ grows
(the figure).

![Log-sum-exp as a soft maximum: $\mathrm{lse}(0,t)=\log(1+e^t)$ stays within $\log 2$ of $\max(0,t)$, matching the sandwich bound for $n=2$. The gap peaks at $t=0$, where the two logits tie, and vanishes as $|t|$ grows.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-softplus-max.svg)

Moreover, its gradient is the softmax itself,
$\partial\,\mathrm{lse}(\mathbf z)/\partial z_i=e^{z_i}/\sum_j e^{z_j}=\mathrm{softmax}(\mathbf z)_i$.
That is why the gradient of cross-entropy with respect to the logits, for
label $y$, simplifies to $\mathrm{softmax}(\mathbf z)-\mathbf e_y$;
that section also shows that log-sum-exp is convex.

Subtracting the shifted log-sum-exp from the shifted logit yields the
**log-softmax**, computing log-probabilities directly without ever forming
the probabilities themselves:

$$
\log \mathrm{softmax}(\mathbf{z})_i = (z_i - c) - \log \sum_{j=1}^n e^{z_j - c},
\qquad c=\max_j z_j .
$$

Because we never exponentiate the final result, a tiny probability cannot
underflow to zero before its logarithm is taken. Notice a subtle detail in
the equation: the algebraically equivalent form
$z_i-\mathrm{lse}(\mathbf z)$ is noticeably worse in floating point, because
it first forms the large number $\mathrm{lse}=c+\log\sum_j e^{z_j-c}$ and
then subtracts it from the large number $z_i$. How much difference does that
parenthesization make? Let us compare both forms in fp32 on the logits
$(1000, 1001, 1002)$:

In [ ]:
z = np.array([1000.0, 1001.0, 1002.0], dtype=np.float32)
c = z.max()
s = np.log(np.exp(z - c).sum())           # log sum exp(z - c), in [0, log 3]
with np.errstate(over='ignore'):
    print('naive log(sum(exp(z))):', np.log(np.exp(z).sum()))
lse = c + s
print('lse = c + s           :', lse, '  float32 gap at 1002:', np.spacing(c))
log_p_sub = z - lse                       # forms lse, then subtracts it
log_p = (z - c) - s                       # never forms c + s
print(f'sum of exp(z - lse)    : {np.exp(log_p_sub).sum():.8f}')
print(f'sum of exp((z - c) - s): {np.exp(log_p).sum():.8f}')

The direct computation overflows to `inf`—as it would even in fp64, since
$1000>709.78$, which is what turned `inf/inf` into three `NaN`s in the
opening. The shifted value $\mathrm{lse}=1002.4076$ sits comfortably inside
the sandwich $[1002, 1002+\log 3]$. More interestingly, the two log-softmax
forms disagree: exponentiating $z_i-\mathrm{lse}$ gives probabilities that
sum to $1.0000132$, whereas the equation sums to
$1.0000000$.

Why did $z_i-\mathrm{lse}$ lose accuracy? Surprisingly, the subtraction
itself made no error at all. Whenever two floating-point numbers lie within a
factor of two of each other, their difference is exactly representable,
because both are multiples of the smaller number's spacing and their
difference needs no extra bits (Sterbenz's lemma, Exercise
6).

The error entered earlier, when $\mathrm{lse}$ was stored. Near $1002$,
fp32 numbers are $2^{-14}\approx6.1\times10^{-5}$ apart, so the true value
$1002.4076060$ rounds down by $1.32\times10^{-5}$. Every log-probability
computed as $z_i-\mathrm{lse}$ is therefore $1.32\times10^{-5}$ too high,
every resulting probability is inflated by the factor $e^{1.32\times10^{-5}}$,
and their sum overshoots $1$ by $1.32\times10^{-5}$. An absolute error in an
exponent turns into a relative error of the same size after exponentiation,
and the subtraction merely exposed an error that was already sitting in its
input—the central theme of that section.

### Cross-Entropy from Logits

The cross-entropy loss for label $y$ is simply the negative log-softmax, so
the equation gives it directly from the logits:

$$
-\log \mathrm{softmax}(\mathbf{z})_y = \mathrm{lse}(\mathbf{z}) - z_y .
$$

Computing the loss this way requires one stable log-sum-exp and one
subtraction, without ever forming a probability. On our running example, the
loss for class $0$ is $\mathrm{lse}(0,t)-0=\log(1+e^t)$. If we instead
compute probabilities first, the true-class probability
$e^{-t}/(1+e^{-t})\approx e^{-t}$ drops below fp32's normal range once
$t>87.3$ and underflows completely to zero near $t=104$, sending its negative
logarithm to $\infty$. Let us compare the two routes using the library's
built-in functions, with $t=103$ and $104$ straddling that cliff:

In [ ]:
print('    t  CE from logits    CE via probabilities')
for t in [20.0, 60.0, 103.0, 104.0]:
    logits = jnp.array([[0.0, t]])
    y = jnp.array([0])                           # label = the unlikely class
    from_logits = optax.softmax_cross_entropy_with_integer_labels(logits, y)
    from_probs = -jnp.log(jax.nn.softmax(logits)[0, 0])
    print(f'{t:5.0f}  {float(from_logits[0]):15.4f}  {float(from_probs):15.4f}')

Computed directly from logits, the loss equals $t$ in every row, and the
probability route matches it at $t=20$ and $60$. At $t=103$, however, the
probability $e^{-103}$ falls into the subnormal range, and what happens next
depends on how the library handles subnormal numbers.

XLA runs compiled operations with subnormals flushed to zero, so the
probability becomes $0$ and the loss jumps to `inf` at both $t=103$ and $104$.

The practical takeaway is simple: keep losses, likelihoods, and posteriors
in log space until you genuinely need a probability. The fused loss in
that section implements
the equation for this reason,
that section interprets what cross-entropy measures,
and that section sums hundreds of per-pixel
log-probabilities to avoid the same underflow.

## Catastrophic Cancellation

### The Condition Number of a Function

In fp32, evaluating $\log(1+x)$ at $x=10^{-8}$ returns $0$—a relative error
of 100%—whereas the library function `log1p` returns $10^{-8}$. Is
$\log(1+x)$ near $x=0$ inherently hard to compute, or is the formula to
blame? To answer questions like this systematically, we need a quantity that
measures how strongly a function amplifies errors in its input, independent
of any particular algorithm.

The **relative condition number** of a differentiable function $f$ at a point
$x$ with $f(x)\ne0$ is the factor by which a small relative perturbation in
$x$ turns into a relative perturbation in $f(x)$. Perturbing $x$ to
$x(1+\delta)$ changes the output, to first order, to
$f(x(1+\delta))\approx f(x)+xf'(x)\,\delta$. Dividing the change by $f(x)$
gives the amplification factor

$$
\kappa_f(x)=\left|\frac{x\,f'(x)}{f(x)}\right| .
$$

For example, the exponential has $\kappa_f(x)=|x|$: at $x=88$, an input error
of one unit roundoff becomes $88$ units in the output. That is the exact
mechanism we just saw in the log-softmax cell, where an absolute error in an
exponent became a relative error of the same size in the probability. Notice
that $\kappa_f$ is a first-order property of the exact mathematical function.
It says nothing about how $f$ is evaluated, and it breaks down where
$f(x)=0$, since relative error is undefined there.

Subtraction takes two inputs, and the same first-order reasoning applies to
both. Suppose $\hat a=a(1+\delta_a)$ and $\hat b=b(1+\delta_b)$ carry
relative errors $|\delta_a|,|\delta_b|\le u$, and suppose the subtraction
itself is exact. Then $\hat a-\hat b=(a-b)+(a\delta_a-b\delta_b)$, so the
triangle inequality bounds the relative error of the difference by

$$
\frac{|(\hat a-\hat b)-(a-b)|}{|a-b|}
=\frac{|a\delta_a-b\delta_b|}{|a-b|}
\le \frac{|a| + |b|}{|a - b|}\; u .
$$

The factor $(|a|+|b|)/|a-b|$ is subtraction's condition number, and it blows
up whenever $a\approx b$. Subtracting two nearly equal numbers that already
carry rounding errors is called **catastrophic cancellation**: the accurate
leading digits cancel out, promoting the noise in the trailing digits into
the leading digits of the answer. For reference, the table below collects the
condition numbers of several common problems:

| problem | $\kappa_f$ | regime |
|----|----|----|
| $a-b$ | $(\lvert a\rvert+\lvert b\rvert)/\lvert a-b\rvert$ | large when $a\approx b$ |
| $\log(1+x)$ | $x/((1+x)\log(1+x))$ | $1.000$ at $x=10^{-8}$, $0.82$ at $x=0.5$ |
| $e^x$ | $\lvert x\rvert$ | large for large $\lvert x\rvert$ |
| $1-\cos x$ | $x\sin x/(1-\cos x)$ | tends to $2$ as $x\to0$ |
| $\sqrt x$ | $1/2$ | always |
| variance of data with mean $m$, spread $s$ | at most $2(1+\lvert m\rvert/s)$ | derived below |

The second row settles our opening question: $\log(1+x)$ near $x=0$ is
completely well conditioned ($\kappa_f\approx 1$). The naive algorithm
`log(1 + x)` fails because it takes a detour through an ill-conditioned
intermediate step. It first forms $y=1+x$ and then evaluates $\log y$ at
$y\approx 1$, where the logarithm has
$\kappa_{\log}(y)=1/|\log y|\approx 1/x=10^8$. Rounding $1+x$ introduces a
relative error of order $u$ in $y$, which $\log$ magnifies by $10^8$—more
than 100% in fp32. By contrast, `log1p` is written so that the rounding error
of $1+x$ never enters the result; for tiny $x$, it simply returns $x$.

This gives us the organizing principle of the entire section: *an unstable
algorithm routes a well-conditioned problem through an ill-conditioned
intermediate step.* Whenever $\kappa_f$ is small and the computed answer is
bad, the algorithm is at fault and we should rewrite it. Whenever $\kappa_f$
itself is large, no algorithm can rescue the answer—we must change the
problem instead, which is the subject of
that section.

### Subtraction of Nearly Equal Numbers

Let us test this in code. The cell below repeats the $\log(1+x)$ experiment
and then subtracts two fp32 numbers that agree in their first seven decimal
digits, $1.0002344$ and $1.0002341$, whose true difference is
$3.0\times10^{-7}$:

In [ ]:
x = np.float32(1e-8)
print('float32 rounds 1 + x to:', np.float32(1.0) + x)
print('log(1 + x) =', np.log(np.float32(1.0) + x), '   log1p(x) =', np.log1p(x))
a, b = np.float32(1.0002344), np.float32(1.0002341)
print(f'stored a = {float(a):.10f}, b = {float(b):.10f}')
print('a - b =', a - b, '=', (a - b) / np.spacing(np.float32(1.0)), 'ulps of 1')
print('float64: log(1 + 1e-20) =', np.log(1 + 1e-20),
      '  log1p(1e-20) =', np.log1p(1e-20))

Between $1$ and $2$, every fp32 difference is an integer multiple of
$\varepsilon_{\mathrm{mach}}=1.19\times10^{-7}$, and the true difference
$3.0\times10^{-7}$ does not fall on that grid. The subtraction returned two
ulps, $2.38\times10^{-7}$—20% too small—yet once again it committed no error
of its own. The inputs were stored as $1.0002343655$ and $1.0002341270$, and
their exact difference is what the cell printed; the error was already baked
into the stored inputs. Here the cancellation factor
the equation is
$2.0005/(3\times10^{-7})\approx7\times10^6$. Multiplied by
$u\approx6\times10^{-8}$, it allows a relative error of up to $0.4$, and the
observed $0.2$ falls squarely inside the bound.

The same cancellation appears in $\log(1+x)$ and $e^x-1$ near $0$ (where
`log1p` and `expm1` avoid it), in $1-\cos x$ near $0$ (rewritten as
$2\sin^2(x/2)$), and in the quadratic formula when $b^2\gg 4|ac|$ (Exercise
2). In every one of
these cases, an algebraic rewrite removes the offending subtraction.

Finite differences are the one place where we cannot eliminate the
subtraction and must settle for a compromise. A numerical gradient check
compares automatic differentiation against the forward difference
$(f(x+h)-f(x))/h$. How small should we make the step $h$? By Taylor's
theorem, the mathematical truncation error is about $\tfrac h2|f''(x)|$,
which shrinks as $h\to 0$. Meanwhile, each function evaluation carries a
rounding error of roughly $u|f(x)|$, and dividing their difference by $h$
magnifies that error to about $2u|f(x)|/h$, which blows up as $h\to 0$. The
total error is minimized where the two terms balance, at

$$
h^\star=2\sqrt{\frac{u\,|f(x)|}{|f''(x)|}},
\qquad \textrm{with error about } 2\sqrt{u\,|f(x)f''(x)|} .
$$

Because the optimal error scales as $\sqrt{u}$, the best possible forward
difference keeps only about half of the format's digits. For $f=\sin$ at
$x=1$, this predicts $h^\star\approx2\times10^{-8}$ (error
$1.8\times10^{-8}$) in fp64 and $h^\star\approx5\times10^{-4}$ in fp32. Let
us sweep $h$ from $10^{-2}$ down to $10^{-12}$ in both formats to see the
trade-off directly:

In [ ]:
print('      h   error in fp64   error in fp32')
for h in 10.0 ** -np.arange(2, 13):
    err = [abs(float((np.sin(t(1) + t(h)) - np.sin(t(1))) / t(h)) - np.cos(1.0))
           for t in (np.float64, np.float32)]  # forward difference of sin at 1
    print(f'{h:7.0e}  {err[0]:14.1e}  {err[1]:14.1e}')

In fp64, the error drops linearly with $h$ from $4\times10^{-3}$ at
$h=10^{-2}$ (where truncation dominates) down to roughly $10^{-8}$ near
$h=10^{-8}$ (the exact value at any single $h$ fluctuates with rounding
noise), and then climbs back to $4\times10^{-5}$ at $h=10^{-12}$ as
cancellation takes over. In fp32, the sweet spot at $h=10^{-3}$ to $10^{-4}$
leaves only about three correct digits; from $h=10^{-8}$ downward, $1+h$
rounds straight to $1$, making the difference quotient $0$ and the error
equal to the entire derivative, $\cos 1=0.54$. That is why numerical gradient
checks should always run in fp64.

Notice, too, that switching to higher precision never cures cancellation—it
merely pushes the failure to smaller inputs. The last line of the earlier
`log1p` cell showed `log(1 + 1e-20)` returning $0$ in fp64 just as fp32 did
at $10^{-8}$, while `log1p` stayed accurate in both. Reformulating the
expression removes the ill-conditioned step itself [@Higham.2002].

### One-Pass Variance

Catastrophic cancellation also hides inside formulas that look completely
innocent. Computing a sample variance is the classic example—and it matters
directly in deep learning, where normalization layers and data-preprocessing
pipelines compute variances continuously.

For data $x_1,\ldots,x_N$ with mean $m$ and standard deviation $s$, the
textbook identity

$$
\frac1N\sum_{i=1}^N(x_i-m)^2 = \frac1N\sum_{i=1}^N x_i^2 - m^2 ,
$$

tempts us to compute the variance in a single pass by accumulating only the
sums of $x_i$ and $x_i^2$. In exact arithmetic the identity is flawless; in
floating point it is a trap. Whenever $s\ll|m|$, both terms on the right sit
near $m^2$ while their difference is $s^2$, so the cancellation factor
the equation is roughly $2m^2/s^2$. For $m=10^9$
and $s=1$, that factor reaches $2\times10^{18}$—two orders of magnitude
larger than the $10^{16}$ that fp64 can absorb.

How much of this loss is forced by the problem itself? Let us check the
condition number of the variance. Perturb each input $x_i$ by a relative
error $\delta_i$ with $|\delta_i|\le u$, replacing $x_i$ by
$x_i+x_i\delta_i$. The derivative of the variance with respect to $x_i$ is
$\tfrac2N(x_i-m)$, because the induced change in $m$ multiplies
$\sum_i(x_i-m)=0$. Writing $x_i=(x_i-m)+m$, the first-order change in the
variance is therefore

$$
\frac2N\sum_{i=1}^N(x_i-m)\,x_i\delta_i
=\frac2N\sum_{i=1}^N(x_i-m)^2\delta_i+\frac{2m}N\sum_{i=1}^N(x_i-m)\,\delta_i .
$$

The first sum has magnitude at most $2s^2 u$. By the Cauchy–Schwarz
inequality, the second has magnitude at most
$2|m|u\cdot\frac1N\sum_i|x_i-m|\le 2|m|s\,u$. Dividing by $s^2$ shows that
the relative condition number of the variance is at most
$2(1+|m|/s)\approx 2|m|/s$, or about $2\times10^9$ here (choosing
$\delta_i=u\,\mathrm{sign}(x_i-m)$ gives $1.6\,|m|/s$ for Gaussian data).
The problem itself therefore permits an error of only
$2\times10^9 u\approx 2\times10^{-7}$—about seven accurate digits. The naive
one-pass formula, by contrast, suffers an error of order $(m/s)^2 u$: it
*squares* the condition number, just as the normal equations will square the
condition number of a matrix in that section.

A stable one-pass algorithm keeps all deviations at the scale $s$ rather than
$m^2$. @Welford.1962 maintains a running mean $m_k$ and a running sum
of *centered* squares $M_k = \sum_{i \le k} (x_i - m_k)^2$, updating both
from the deviation $d_k=x_k-m_{k-1}$ of each new sample from the previous
mean:

$$
m_k = m_{k-1} + \frac{d_k}{k},
\qquad
M_k = M_{k-1} + d_k\,(x_k - m_k).
$$

Notice the asymmetry in the update of $M_k$: it multiplies the deviation from
the *old* mean by the deviation from the *new* mean. That asymmetry is
precisely what makes the recursion exact:

**Proposition (Welford's recursion).** *With $m_0 = M_0 = 0$, the
recursions the equation satisfy, for every $k \ge 1$ and in
exact arithmetic,*

$$
m_k = \frac{1}{k} \sum_{i=1}^k x_i,
\qquad
M_k = \sum_{i=1}^k (x_i - m_k)^2 .
$$

**Proof.** The mean identity $k\, m_k = (k-1)\, m_{k-1} + x_k$ follows
immediately from the first recursion. For $M_k$, we induct on $k$. Using
$m_k - m_{k-1} = d_k/k$ and $x_k - m_k = d_k\,(k-1)/k$, we split off the
$k$-th term and recenter the first $k-1$ terms around $m_{k-1}$:

$$
\sum_{i=1}^{k} (x_i - m_k)^2
= \sum_{i=1}^{k-1} \left( (x_i - m_{k-1}) - \tfrac{d_k}{k} \right)^2 + (x_k - m_k)^2
= M_{k-1} + (k-1)\tfrac{d_k^2}{k^2} + \tfrac{(k-1)^2}{k^2}d_k^2 .
$$

The cross term vanishes because $\sum_{i \le k-1} (x_i - m_{k-1}) = 0$, and
the inductive hypothesis identifies the first sum as $M_{k-1}$. Combining the
remaining two terms gives
$\frac{k-1}{k}\,d_k^2 = d_k \cdot d_k\frac{k-1}{k} = d_k(x_k - m_k)$, which
is exactly what the recursion adds. $\blacksquare$

Proving that Welford's recursion also stays accurate in floating point is the
harder step, and we sketch only the intuition here (see @Higham.2002
for the full analysis). At first glance, Welford's update still subtracts
large numbers, since $x_k$ and $m_{k-1}$ both sit near $10^9$. By Sterbenz's
lemma, however, that subtraction is exact, and $x_k$ is exact input data, so
the only error in $d_k$ comes from the running mean—at most a few dozen ulps
at $|m|$ over $10^5$ steps. Every quantity added into $M_k$ is therefore a
deviation of scale $s$, so rounding errors in the updates enter relative to
$s^2$ rather than $m^2$. Even a constant offset $\varepsilon$ in the mean
shifts a centered sum of squares only by $N\varepsilon^2$, because the
first-order term multiplies $\sum_i(x_i-m)=0$. In practice, Welford's error
matches the problem's own sensitivity, $u|m|/s\approx 1.1\times10^{-7}$,
whereas the naive one-pass formula suffers an error of
$(m/s)^2 u\approx 100$.

Let us compare all three methods—naive one-pass, Welford, and a two-pass
reference—on $10^5$ fp64 samples with mean $10^9$ and variance $1$:

In [ ]:
rng = np.random.default_rng()
x = 1e9 + rng.normal(0.0, 1.0, size=100_000)    # mean 1e9, variance 1

naive = (x**2).mean() - x.mean()**2             # one pass, cancels
two_pass = ((x - x.mean())**2).mean()           # subtract the mean first

m, M = 0.0, 0.0                                 # Welford: one pass
for k, xk in enumerate(x, start=1):
    d = xk - m
    m += d / k                                  # m_k
    M += d * (xk - m)                           # M_k
welford = M / len(x)

print(f'naive E[x^2] - E[x]^2 : {naive:12.6f}   (= {naive / 128:.0f} x 128)')
print(f'Welford, one pass     : {welford:12.6f}')
print(f'two-pass reference    : {two_pass:12.6f}')
print(f'Welford vs two-pass   : {abs(welford - two_pass) / two_pass:.1e} relative')
print('float64 gap near 1e18 :', np.spacing(1e18))

The naive one-pass estimate comes out as an exact multiple of $128$—typically
$0$, $\pm128$, or $\pm256$—nowhere near the true variance $1$, and frequently
negative. Why $128$? Both $\frac1N\sum x_i^2$ and $m^2$ lie in
$[2^{59}, 2^{60})$, where adjacent fp64 numbers are $2^{59-52}=128$ apart, so
their difference can only be a multiple of $128$: our predicted error of
about $100$, quantized to the grid. Welford's recursion, on the other hand,
matches the two-pass reference to at least seven digits—and often eight or
nine, because rounding errors of opposite signs partly cancel across the sum.

The same caution applies whenever a variance is accumulated in one pass, such
as computing streaming statistics over a dataset. (The running moments that
batch normalization maintains for inference are exponential moving averages,
that section, which do not subtract large running sums.)

### Summation Order

Both the naive variance and the two-pass reference sum $10^5$ numbers, and in
floating point the order of addition matters. Each addition introduces a
factor $(1+\delta_k)$, so in a plain left-to-right sum the first term passes
through all $n-1$ roundings. To first order in $u$, a left-to-right sum
satisfies

$$
\Bigl|\mathrm{fl}\Bigl(\sum_{i=1}^n x_i\Bigr)-\sum_{i=1}^n x_i\Bigr|
\le(n-1)\,u\sum_{i=1}^n|x_i| .
$$

Because early terms accumulate the rounding error of every subsequent
addition, the worst-case error grows linearly with $n$. *Pairwise* summation
avoids this by recursively summing the two halves of the array, so every term
passes through only $\log_2 n$ additions and the bound shrinks to
$u\log_2 n\sum_i|x_i|$; NumPy's `sum` uses a blocked version of this idea.
Kahan's compensated summation goes one step further, carrying each addition's
rounding error in a running correction variable to achieve an error bound
independent of $n$ (Exercise
8). Let
us add $10^7$ copies of fp32's $0.1$ (whose stored value is
$0.100000001490\ldots$) both left to right and pairwise, padding the pairwise
array with zeros to $2^{24}$ entries and repeatedly adding adjacent pairs:

In [ ]:
x = np.full(10_000_000, 0.1, dtype=np.float32)
exact = len(x) * float(x[0])                  # x[0] = 0.100000001490...
s = np.concatenate([x, np.zeros(2**24 - len(x), np.float32)])  # pad to 2^24
while len(s) > 1:                             # pairwise: add neighbors, halve
    s = s[0::2] + s[1::2]
print(f'in order (cumsum): {np.cumsum(x)[-1]:16.4f}')
print(f'pairwise         : {s[0]:16.4f}')
print(f'exact            : {exact:16.4f}')

The left-to-right sum finishes at $1{,}087{,}937$—almost 9% too high, though
still inside its bound $(n-1)u\sum_i|x_i|\approx6\times10^5$. Pairwise
summation lands at $1{,}000{,}000$, within $0.015$ of the exact sum and well
inside its bound $u\log_2 n\sum_i|x_i|\approx1.4$.

The culprit behind the left-to-right drift is the same grid effect that
erased small weight updates: each addend is rounded to the spacing of the
current running sum. Once the sum crosses $2^{19}=524{,}288$, fp32 numbers
sit $1/16$ apart, so $0.1$ equals $1.6$ gaps and every addition rounds up to
two gaps, $0.125$—25% too large. Above $2^{20}$ the spacing doubles to $1/8$,
and each addend again rounds to $0.125$; earlier, between $2^{17}$ and
$2^{19}$, it rounded down to $0.09375$. Because floating-point addition is
not associative, a parallel reduction across GPU threads need not produce the
exact same bits from one device or run to another. Fortunately, Welford's
recursion also supports pairwise reduction: the merge rule in Exercise
7 combines
statistics from two partial sums, which is how running moments are merged
across devices in distributed training.

## Conditioning

The condition number of a matrix and the filter factors of ridge regression
are developed in that section. New here are forward and
backward error, a formula that measures the backward error, and the price in
digits of the normal equations.

### Two Nearly Parallel Lines

So far the answer of a computation was a number. For a linear system it is a
vector, and that section showed on the system

$$
x_1+x_2=2, \qquad x_1+1.0001\,x_2=2.0001 ,
$$

how fragile such an answer can be. The solution is $(1,1)$, and raising the
second right-hand side by $10^{-4}$, a relative change of $3.5\times10^{-5}$
in the data, moves it to $(0,2)$. Each equation is a line in the plane, and
the solution is where the lines cross. These two lines have slopes $-1$ and
$-0.9999$ and meet at an angle of $0.003^\circ$, so shifting one of them
slightly slides the crossing far along the other
(the figure).

![A linear system is sensitive when its lines are nearly parallel. (a) Two perpendicular lines meet at $(1,1)$; shifting the second by $0.1$ moves the crossing by 5%: amplification $1$, $\kappa=1$. (b) The lines $x_1+x_2=2$ and $x_1+1.1x_2=2.1$ also meet at $(1,1)$, but the same shift moves the crossing to $(0,2)$, a 100% change from a 3.4% change in $\mathbf{b}$: amplification $29$, bounded by $\kappa\approx42$. The text's system, with $1.0001$ in place of $1.1$, has the same geometry at an angle of $0.003^\circ$ and $\kappa\approx4\times10^4$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-two-lines.svg)

Let us read the same numbers in the other direction. Suppose a solver returned
$\hat{\mathbf x}=(0,2)$ for the original system. Its **forward error**, the
distance to the true solution, is 100%. Yet $(0,2)$ is the exact solution of a
nearby problem, the one with $b_2$ changed by $10^{-4}$. The smallest such
change of the data is the **backward error**. A solver with a tiny backward
error has solved a nearby problem exactly, and it can still be far from the
answer.

How far is a property of the problem, not of the solver. It is the condition
number of that section once more, now for
the function $\mathbf b\mapsto\mathbf A^{-1}\mathbf b$: the largest factor
by which a relative change of the input can become a relative change of the
output, taken over all right-hand sides and all directions of the change.
For a matrix with singular values $\sigma_1\ge\cdots\ge\sigma_d>0$ that
factor is $\kappa(\mathbf A)=\sigma_1/\sigma_d$, by
the equation, and it is about $4\times10^4$ for the two
lines. The shift $(0,10^{-4})$ realizes $2.8\times10^4$ of it, because it is
not along the most amplified direction.

### Backward and Forward Error

For a linear system $\mathbf A\mathbf x=\mathbf b$ and a computed solution
$\hat{\mathbf x}$, the forward error is
$\|\hat{\mathbf x}-\mathbf x\|/\|\mathbf x\|$ and the backward error is the
size of the smallest change of the data that makes $\hat{\mathbf x}$ exact.
An algorithm is **backward stable** if its backward error is of the order of
$u$, so that it returns the exact answer of a problem that differs from the
given one by about one rounding of the data. The SVD and the QR decomposition
are backward stable, and so in practice is the LU decomposition with pivoting
that `np.linalg.solve` uses. The one-pass variance is not: the variance of
any data set is nonnegative, so a negative output is the exact variance of no
data at all.

The definition asks for a minimization over all changes of the data. Luckily,
for changes of the matrix it has a closed form, computable from the residual.

**Proposition (backward error from the residual).** *Let
$\hat{\mathbf x}\ne\mathbf 0$ and $\mathbf r=\mathbf b-\mathbf A\hat{\mathbf x}$.
The smallest $\|\Delta\mathbf A\|_2$ with
$(\mathbf A+\Delta\mathbf A)\hat{\mathbf x}=\mathbf b$ is
$\|\mathbf r\|/\|\hat{\mathbf x}\|$, attained by
$\Delta\mathbf A=\mathbf r\hat{\mathbf x}^\top/\|\hat{\mathbf x}\|^2$.*

**Proof.** Any such $\Delta\mathbf A$ satisfies
$\Delta\mathbf A\hat{\mathbf x}=\mathbf b-\mathbf A\hat{\mathbf x}=\mathbf r$,
so $\|\mathbf r\|\le\|\Delta\mathbf A\|\,\|\hat{\mathbf x}\|$. The rank-one
matrix $\mathbf r\hat{\mathbf x}^\top/\|\hat{\mathbf x}\|^2$ maps
$\hat{\mathbf x}$ to $\mathbf r$, and its norm is
$\|\mathbf r\|\,\|\hat{\mathbf x}\|/\|\hat{\mathbf x}\|^2=\|\mathbf r\|/\|\hat{\mathbf x}\|$.
$\blacksquare$

Relative to the size of $\mathbf A$, the backward error is

$$
\omega=\frac{\|\mathbf b-\mathbf A\hat{\mathbf x}\|}{\|\mathbf A\|\,\|\hat{\mathbf x}\|} .
$$

A small residual certifies a small backward error, whatever the condition of
the problem. For $\hat{\mathbf x}=(0,2)$ in the two-line system,
$\mathbf r=(0,-10^{-4})$ and
$\omega=10^{-4}/(2.00005\cdot2)=2.5\times10^{-5}$. The version that perturbs
$\mathbf b$ as well is due to Rigal and Gaches (see @Higham.2002). In
floating point the residual is itself computed with rounding errors, so
printed values of $\omega$ below about $u$ are noise.

The condition number converts backward error into forward error. This is the
result to remember, and its proof needs only the two norms
$\|\mathbf A\|=\sigma_1$ and $\|\mathbf A^{-1}\|=1/\sigma_d$
(that section).

**Proposition (forward error $\le$ condition number $\times$ backward
error).** *Let $\mathbf{A}$ be invertible, let
$\mathbf{A}\mathbf{x} = \mathbf{b}$, and suppose the computed
$\hat{\mathbf{x}}$ exactly solves a nearby system,
$(\mathbf{A} + \Delta\mathbf{A})\,\hat{\mathbf{x}} = \mathbf{b}$ with
$\|\Delta\mathbf{A}\| \le \omega \|\mathbf{A}\|$. Then*

$$
\frac{\|\hat{\mathbf{x}} - \mathbf{x}\|}{\|\hat{\mathbf{x}}\|}
\;\le\; \kappa(\mathbf{A})\, \omega .
$$

**Proof.** Subtracting $\mathbf{A}\mathbf{x} = \mathbf{b}$ from
$(\mathbf{A} + \Delta\mathbf{A})\hat{\mathbf{x}} = \mathbf{b}$ gives
$\mathbf{A}(\hat{\mathbf{x}} - \mathbf{x}) = -\Delta\mathbf{A}\,\hat{\mathbf{x}}$,
hence $\hat{\mathbf{x}} - \mathbf{x} = -\mathbf{A}^{-1}\Delta\mathbf{A}\,\hat{\mathbf{x}}$
and

$$
\|\hat{\mathbf{x}} - \mathbf{x}\|
\le \|\mathbf{A}^{-1}\|\, \|\Delta\mathbf{A}\|\, \|\hat{\mathbf{x}}\|
\le \frac{1}{\sigma_d}\, \sigma_1\, \omega\, \|\hat{\mathbf{x}}\|
= \kappa(\mathbf{A})\,\omega\, \|\hat{\mathbf{x}}\| .
\qquad\blacksquare
$$

The error here is measured relative to $\hat{\mathbf x}$, the vector we have.
Relative to the true $\mathbf x$, substituting
$\|\hat{\mathbf x}\|\le\|\mathbf x\|+\|\hat{\mathbf x}-\mathbf x\|$ and
solving for $\|\hat{\mathbf x}-\mathbf x\|$ gives the bound
$\kappa\omega/(1-\kappa\omega)$ when $\kappa\omega<1$, the same to first
order. On the two-line system, $\kappa\omega=4\times10^4\cdot
2.5\times10^{-5}=1.0$, and the error relative to $\hat{\mathbf x}$ is
$\|(-1,1)\|/\|(0,2)\|=0.71$, inside the bound. For an error in the
right-hand side instead of the matrix, the corresponding bound is
the equation.

A backward-stable solver has $\omega\approx u$, and taking $\log_{10}$ of the
bound gives the rule of thumb

$$
\textrm{correct digits in } \hat{\mathbf{x}}
\;\approx\; \textrm{digits carried by the format} \;-\; \log_{10} \kappa(\mathbf{A}).
$$

The rule is a worst case. A matrix with $\kappa=10^3$ can lose three of fp32's
seven digits. The chapter's running example $\tfrac12(x_1^2+1000x_2^2)$ has
that $\kappa$ and loses none: its matrix is diagonal, and each unknown takes
one correctly rounded division.

### Example: The Hilbert Matrix

The **Hilbert matrix** $\mathbf H_{ij}=1/(i+j-1)$, $i,j=1,\ldots,n$, has a
condition number that grows exponentially with $n$: its rows become nearly
parallel in the same way as the two lines above. We solve
$\mathbf H\mathbf x=\mathbf b$ with $\mathbf b=\mathbf H\mathbf 1$, whose
exact solution is $\mathbf 1$, and compare the forward error with the rule of
thumb and with the backward error the equation.
One caveat before we judge the solver. Computing
$\mathbf b=\mathbf H\mathbf 1$ in fp64 rounds every entry of $\mathbf b$, a
change of the data of relative size $u$ before the solver starts. (Rounding
the entries $1/3$, $1/7$, … does no harm here, because $\mathbf b$ is built
from the matrix as stored.) How much does that alone cost? Every fp64
number is a rational number, so the stored system can also be solved without
any rounding, by Gauss–Jordan elimination on `Fraction` entries. The table
reports the error of that exact solve beside the error of `np.linalg.solve`.

In [ ]:
def solve_exact(A, b):
    """Gauss-Jordan elimination in exact rational arithmetic."""
    M = [[Fraction(a) for a in row] + [Fraction(c)] for row, c in zip(A, b)]
    n = len(M)
    for j in range(n):              # no pivoting: A is positive definite here
        for i in range(n):
            if i != j:
                r = M[i][j] / M[j][j]
                M[i] = [a - r * c for a, c in zip(M[i], M[j])]
    return np.array([float(M[i][n] / M[i][i]) for i in range(n)])

In [ ]:
print(' n     kappa  forward error  digits kept  16 - log10 kappa  backward error'
      '  exact solve error')
for n in [4, 6, 8, 10, 12]:
    i = np.arange(n)
    H = 1.0 / (1.0 + i[:, None] + i[None, :])   # Hilbert matrix
    x_true = np.ones(n)
    b = H @ x_true
    x_hat = np.linalg.solve(H, b)
    kappa = np.linalg.cond(H)
    fwd = np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true)
    omega = (np.linalg.norm(b - H @ x_hat)
             / (np.linalg.norm(H, 2) * np.linalg.norm(x_hat)))
    exact = np.linalg.norm(solve_exact(H, b) - x_true) / np.linalg.norm(x_true)
    print(f'{n:2d}  {kappa:8.1e}  {fwd:13.1e}  {-np.log10(fwd):11.1f}'
          f'  {16 - np.log10(kappa):16.1f}  {omega:14.1e}  {exact:17.1e}')

The digits kept fall from about $13$ at $n=4$ to well under one at $n=12$,
while the backward error stays at or below about $10^{-16}$: each computed
vector solves a system that differs from the stored one in the sixteenth or
seventeenth digit. Is the solver to blame for the lost digits, then? The
last column acquits it. The exact solve of the stored system, which commits
no rounding error at all, misses $\mathbf 1$ by an error of the same order
in every row, about $10^{-7}$, $10^{-4}$ and $10^{-1}$ for $n=8$, $10$ and
$12$. The digits were lost when $\mathbf b$ was stored, before any solver
ran. The rule of
thumb is pessimistic by about one digit (from about half a digit to about
one and a half, depending on the row), because the error is not aligned with
the most amplified direction. At
$n=12$, $\kappa u\approx2$ and the bound no longer constrains anything. The
last decimals vary with the LAPACK build; the trend does not.

### Conditioning of the Normal Equations

The least-squares problem
$\min_{\mathbf{w}}\|\mathbf{A}\mathbf{w}-\mathbf{b}\|^2$ with
$\mathbf A\in\mathbb R^{n\times d}$ of full column rank is solved by the
normal equations
$\mathbf{A}^\top\mathbf{A}\mathbf{w}=\mathbf{A}^\top\mathbf{b}$. This is an
algebraically equivalent route through a worse-conditioned matrix
(that section):

$$
\kappa(\mathbf{A}^\top\mathbf{A}) = \kappa(\mathbf{A})^2 .
$$

We have seen this before, in the one-pass variance: the convenient formula
squares the condition number of the problem. By the rule of thumb, the normal
equations can lose about $2\log_{10}\kappa$ digits. A solve by QR
($\mathbf A=\mathbf Q\mathbf R$ with orthonormal $\mathbf Q$ and triangular
$\mathbf R$) or by the SVD works on $\mathbf A$ directly and loses about
$\log_{10}\kappa$, *when the residual $\mathbf b-\mathbf A\mathbf w$ is
small*. With a large residual the least-squares problem itself is more
sensitive: its condition number is about $\kappa+\kappa^2\tan\theta$, where
$\theta$ is the angle between $\mathbf b$ and the column space of $\mathbf A$.
QR and SVD then lose close to $2\log_{10}\kappa$ digits too, but never more
than the problem forces; the normal equations lose that many whatever the
residual.

The chapter's running example is too well conditioned to show the difference:
its design has $\mathbf A^\top\mathbf A=\mathrm{diag}(1,1000)$, so
$\kappa(\mathbf A)=\sqrt{1000}\approx32$. The cell therefore builds harder
designs, with $\kappa(\mathbf A)=10^3$ and $10^6$, from random orthonormal
factors and a right-hand side in the column space (zero residual). We expect
gaps of three and of six digits:

In [ ]:
rng = np.random.default_rng()
n, d = 100, 10
for kappa in [1e3, 1e6]:
    U, _ = np.linalg.qr(rng.normal(size=(n, d)))    # orthonormal columns
    V, _ = np.linalg.qr(rng.normal(size=(d, d)))
    sigma = np.logspace(0, -np.log10(kappa), d)     # kappa(A) as chosen
    A = U * sigma @ V.T                             # A = U diag(sigma) V^T
    w_true = rng.normal(size=d)
    b = A @ w_true                                  # zero residual
    w_ne = np.linalg.solve(A.T @ A, A.T @ b)        # normal equations
    w_ls = np.linalg.lstsq(A, b, rcond=None)[0]     # SVD-based solve
    digits = [-np.log10(np.linalg.norm(w - w_true) / np.linalg.norm(w_true))
              for w in (w_ne, w_ls)]
    print(f'kappa(A) = {kappa:.0e}: normal equations keep {digits[0]:4.1f} '
          f'digits, lstsq keeps {digits[1]:4.1f}')

At $\kappa(\mathbf A)=10^3$ the normal equations keep about $10$ to $12$
digits against about $14$ for `lstsq`, and at $10^6$ about $4$ to $6$
against about $11$ to $13$. The exact digits vary from run to run by a digit
or two; the gaps scatter around $\log_{10}\kappa$, mostly within a digit of
it. This is why `lstsq` and the pseudoinverse of
that section work from the SVD and not from
$(\mathbf A^\top\mathbf A)^{-1}\mathbf A^\top$.

### Ridge Regularization and Conditioning

When $\kappa(\mathbf A)$ itself is large, because features are nearly
collinear or the design is rank deficient, no choice of algorithm helps: the
problem must change. One change keeps the answer. A **preconditioner**
changes variables so that the new problem is better conditioned and its
solution maps back to the original one. Dividing each column of $\mathbf A$
by its norm is an example (Exercise
4), and the
per-coordinate step sizes of that section
do the same inside an optimizer.

Ridge regularization changes the answer as well. It minimizes
$\tfrac12\|\mathbf{A}\mathbf{w}-\mathbf{b}\|^2+\tfrac\lambda2\|\mathbf{w}\|^2$,
whose minimizer is

$$
\mathbf{w}_\lambda = (\mathbf{A}^\top\mathbf{A} + \lambda \mathbf{I})^{-1} \mathbf{A}^\top \mathbf{b} ,
$$

the ridge solution of that section. Let $\sigma_1\ge\cdots\ge\sigma_d\ge0$ be the singular values
of $\mathbf A\in\mathbb R^{n\times d}$, with $\sigma_i=0$ beyond its rank. The
matrix $\mathbf A^\top\mathbf A+\lambda\mathbf I$ has the eigenvectors of
$\mathbf A^\top\mathbf A$ and the eigenvalues
$\sigma_i^2+\lambda\ge\lambda>0$. It is therefore positive definite, hence
invertible even when $\mathbf A$ is rank deficient, and

$$
\kappa(\mathbf{A}^\top\mathbf{A} + \lambda\mathbf{I})
= \frac{\sigma_1^2 + \lambda}{\sigma_d^2 + \lambda} .
$$

Whenever $\sigma_1>\sigma_d$ this ratio decreases strictly in $\lambda$ (its
derivative is $(\sigma_d^2-\sigma_1^2)/(\sigma_d^2+\lambda)^2<0$) and tends to
$1$ as $\lambda\to\infty$. Adding $\lambda$ lifts the floor of the spectrum.
While $\lambda\ll\sigma_1^2$ it barely moves the ceiling, so $\kappa$ falls
about in proportion to $1/\lambda$ once $\lambda\gg\sigma_d^2$. The damping
term of second-order optimizers and the jitter added to a kernel matrix in
Gaussian process regression (that section) are this $\lambda\mathbf I$
under other names.

The price is a different minimizer. In the present notation, with
$\mathbf v_i$ the right singular vectors of $\mathbf A$ and
$\mathbf w_0=\mathbf A^+\mathbf b$ the minimum-norm least-squares solution,
the filter factors of the equation read

$$
\mathbf w_\lambda=\sum_{i=1}^d\frac{\sigma_i^2}{\sigma_i^2+\lambda}\,(\mathbf v_i^\top\mathbf w_0)\,\mathbf v_i .
$$

Ridge damps most the directions with small $\sigma_i$, and these are the very
directions in which the unregularized solve amplifies data errors by
$1/\sigma_i$. the figure shows the effect on
the level sets; each panel is centered on its own minimizer, so the shift of
the minimizer toward the origin is not drawn.

![Level sets of the least-squares objective before and after adding the ridge term $\tfrac\lambda2\|\mathbf{w}\|^2$, drawn for $\kappa=16$ so that both panels stay legible (the running example's $\kappa=1000$ would draw as a line). The penalty lifts every eigenvalue of $\mathbf{A}^\top\mathbf{A}$ by $\lambda$; the semi-axes, proportional to $1/\sqrt{\sigma_i^2+\lambda}$, are long where the curvature is small, and the valley rounds into a bowl. Orange: gradient-descent iterates from the same start at each panel's best fixed step; green arrows: the principal axes, labeled by their curvatures.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-conditioning-ellipse.svg)

The same $\kappa$ counts the steps of gradient descent on the ridge objective.
At the best fixed step, gradient descent on a quadratic with condition number
$\kappa>1$ contracts the error by $\rho=(\kappa-1)/(\kappa+1)$ per step,
by the equation. Started at the origin, where the error is
$-\mathbf w^\star$, it has relative error
$\|\mathbf w_k-\mathbf w^\star\|/\|\mathbf w^\star\|\le\rho^k$ after $k$
steps, so a tolerance $0<\mathrm{tol}<1$ is met within

$$
k \le \left\lceil\frac{\log(1/\mathrm{tol})}{\log\frac{\kappa+1}{\kappa-1}}\right\rceil
\le \left\lceil\frac\kappa2\,\log\frac1{\mathrm{tol}}\right\rceil
\quad\textrm{steps.}
$$

The second inequality holds because
$\log\frac{\kappa+1}{\kappa-1}=2\,\mathrm{artanh}(1/\kappa)\ge2/\kappa$. For
six digits it allows $\tfrac12\log10^6=6.9$ steps per unit of $\kappa$; a
run in more than two dimensions can finish sooner (Exercise
4).

How many steps does a given $\lambda$ save, and what does it cost? We measure
both on the running example. Its objective
$\tfrac12(w_1^2+1000w_2^2)-w_1-1000w_2$ is least squares with
$\mathbf A^\top\mathbf A=\mathrm{diag}(1,1000)$ and
$\mathbf A^\top\mathbf b=(1,1000)$. Its minimizer is $(1,1)$, so the ridge
minimizer displays the two filter factors directly. The function
`gd_iterations` runs gradient descent at the best fixed step and returns the
step count and the ridge minimizer:

In [ ]:
def gd_iterations(AtA, Atb, lam, tol=1e-6):
    """GD steps at step 2/(L+mu) to reach relative error tol on ridge."""
    M = AtA + lam * np.eye(len(Atb))
    w_star = np.linalg.solve(M, Atb)
    mu, L = np.linalg.eigvalsh(M)[[0, -1]]
    eta, w, k = 2 / (L + mu), np.zeros(len(Atb)), 0
    while np.linalg.norm(w - w_star) > tol * np.linalg.norm(w_star):
        w, k = w - eta * (M @ w - Atb), k + 1
    return k, w_star

AtA, Atb = np.diag([1.0, 1000.0]), np.array([1.0, 1000.0])
print('lambda   kappa  steps  steps/kappa  ridge minimizer')
for lam in [0, 1, 10, 100, 1000]:
    kappa = (1000 + lam) / (1 + lam)
    k, w_star = gd_iterations(AtA, Atb, lam)
    print(f'{lam:6d} {kappa:7.1f} {k:6d} {k / kappa:12.2f}  {w_star.round(4)}')

The condition number falls from $1000$ to $2$ as $\lambda$ goes from $0$ to
$1000$, and the step count falls with it, from $6{,}908$ to $13$. The ratio of
steps to $\kappa$ stays near $6.9$ (at small $\kappa$ the approximation
$2/\kappa$ is loose, and rounding up to a whole step matters). The last column
shows what the saved steps cost. At $\lambda=1$ the slow direction, with
$\sigma^2=1$, is halved to $0.5$, while the stiff direction, with
$\sigma^2=1000$, moves by $0.1\%$. At $\lambda=1000$ the stiff direction is
halved too. A sweep over $\lambda$ shows $\kappa$ and the step count together:

In [ ]:
lams = np.logspace(-2, 3, 11)
kappas = (1000 + lams) / (1 + lams)
steps = [gd_iterations(AtA, Atb, lam)[0] for lam in lams]
d2l.plot(lams, [kappas, steps], 'lambda', None,
         legend=['kappa', 'GD steps to 1e-6'], xscale='log', yscale='log')

On log–log axes the two curves are parallel, a factor of $6.9$ apart, with
slope close to $-1$ while $\lambda$ lies well between the eigenvalues $1$
and $1000$. Ridge thus conditions the linear solve and
gradient descent alike, and it pays for both by shrinking the solution.
that section shows that the penalty
$\tfrac\lambda2\|\mathbf w\|^2$ corresponds to a constraint on
$\|\mathbf w\|$, so a $\lambda$ chosen for statistical reasons also sets the
conditioning of the computation.

None of these tools covers error that grows over many steps of an iteration,
such as a gradient through a hundred layers;
that section takes that up.

## Summary

A floating-point format stores numbers with about $p$ bits of relative
precision and a finite exponent range. Every rounding introduces a relative
error of at most $u=2^{-p-1}$, while overflow, underflow, and a lost small
update are absolute effects that the relative model does not show. Softmax and
cross-entropy fail because they pass through values the format cannot hold,
and the one-pass variance because it subtracts nearly equal large numbers;
none of their answers is extreme. The max shift, cross-entropy computed from
logits, and Welford's recursion avoid the detour. The relative condition
number $\kappa_f=|xf'/f|$ separates a bad algorithm from a hard problem: when
it is small and the computed answer is bad, the algorithm is to blame. For a
linear system, $\kappa(\mathbf A)$ bounds the forward error by $\kappa$ times
the backward error, which a residual certifies, so about $\log_{10}\kappa$
digits are lost, and twice that through the normal equations. Ridge adds
$\lambda$ to every squared singular value, which lowers $\kappa$ for the solve
and for gradient descent alike, at the price of shrinking the solution by the
filter factors $\sigma_i^2/(\sigma_i^2+\lambda)$.

## Exercises

1. **Loss-scale window.** Loss scaling multiplies the loss by a factor $S$,
   which multiplies every gradient by $S$, and divides the gradients by $S$
   after they have been converted to fp32. In fp16 the smallest normal
   number is $2^{-14} \approx 6.1 \times 10^{-5}$, the smallest subnormal
   number is $2^{-24} \approx 6.0 \times 10^{-8}$ (values below $2^{-25}$
   round to zero), and the largest finite number is $65504$. Suppose that
   the exact nonzero gradient entries of a network have magnitudes between
   $10^{-9}$ and $10^{-1}$, and that the backward pass computes the scaled
   gradients in fp16.
    1. Find all powers of two $S$ for which every scaled gradient is a
       normal fp16 number.
    1. The gradients of a deeper network have magnitudes between
       $10^{-12}$ and $10^{-1}$. Show that no scale keeps all of them
       normal. Take the largest power of two $S$ that avoids overflow. How
       many significant bits does a scaled gradient of magnitude $10^{-12}$
       keep, and what is its relative rounding error?
    1. Dynamic loss scaling (that section) picks the
       largest scale that avoids overflow and thereby sacrifices the
       smallest gradients. Why is that the better end to sacrifice?
    1. How do the answers to the first two sub-items change when the
       gradients are computed in bf16?

1. [code] **Blame the problem or the algorithm.** Each expression below is
   evaluated in fp32. For each, compute the relative condition number
   the equation of the *function* in the stated regime and the
   cancellation factor the equation of the
   *formula as written*. Decide whether the problem or the algorithm is to
   blame, and rewrite the formula so that its error matches the problem's.
   Check both forms against fp64.
    1. The root of smaller magnitude of $ax^2 + bx + c = 0$ for $b > 0$ and
       $b^2 \gg 4|ac|$, as a function of $c$, for example $a = c = 1$ and
       $b = 10^4$. (Hint: multiply by the conjugate.)
    1. The function $\log(1 - e^{-x})$ for $x > 0$, which computes
       $\log(1 - q)$ from $\log q = -x$, at
       $x \in \{10^{-6}, 10^{-3}, 1, 20\}$. Find one formulation that is
       accurate for small $x$ and one that is accurate for large $x$, and
       choose a switching point.
    1. The binary cross-entropy of a logit $t$ with label $1$,
       $\log(1+e^{-t})$, at $t\in\{-800,-50,0,50\}$. At which of these
       does the formula as written overflow, and at which does it return
       $0$? Show that $\max(-t,0)+\mathrm{log1p}(e^{-|t|})$ equals it,
       never overflows, and loses no digits. Which part of the log-sum-exp
       sandwich is it?
    1. $\sqrt{x+1}-\sqrt x$ at $x=10^8$.

    *Part 2 is adapted from Martin Mächler, [Accurately Computing log(1 − exp(−|a|))](https://cran.r-project.org/web/packages/Rmpfr/vignettes/log1mexp-note.pdf), Section 2.*

1. [code] **Normal equations in floating point.** The normal equations
   square the condition number the equation.
    1. For $\tau > 0$, the $(m + 1) \times m$ matrix $\mathbf{A}$ of
       @Lauchli.1961 has a first row of ones and $\tau\mathbf{I}$
       below it. Compute $\mathbf{A}^\top\mathbf{A}$, the singular values
       of $\mathbf{A}$, and $\kappa(\mathbf{A})$. Show that for
       $\tau = 10^{-8}$ the matrix $\mathbf{A}^\top\mathbf{A}$ rounds to
       a singular matrix in fp64. How many correct digits does the rule
       of thumb of that section predict for
       a QR-based least-squares solve with this $\mathbf{A}$ when
       $\mathbf b$ lies in the range of $\mathbf A$? What changes in the
       prediction when the residual is large?
    1. Modify the `#numerical-stability-conditioning-normal-equations` cell
       to sweep $\kappa(\mathbf{A}) = 10^k$ for $k = 2, \ldots, 10$. Plot
       the correct digits of the normal equations and of `lstsq` against
       $k$, together with the two rule-of-thumb lines $16-2k$ and $16-k$.
       At which $k$ does the rule predict that the normal equations keep no
       correct digit, and what do you observe there?

1. [code] **Feature scaling as preconditioning.** Features measured in
   different units give the columns of a design matrix very different
   norms. Let $\mathbf{X}$ be a $200 \times 5$ matrix with standard normal
   entries, let $\mathbf{A} = \mathbf{X}\,\mathrm{diag}(1, 3, 10, 30, 100)$,
   and let $\mathbf{b} = \mathbf{A}\mathbf{w}^\star$ for a random
   $\mathbf{w}^\star$.
    1. Apply `gd_iterations` of the `#numerical-stability-conditioning-ridge`
       cell, which takes $\mathbf A^\top\mathbf A$ and $\mathbf A^\top\mathbf b$,
       to this design at $\lambda = 0$. Compute
       $\kappa(\mathbf{A}^\top\mathbf{A})$ and compare the step count with
       the bound the equation. Explain the
       shortfall through the components of the start's error along the two
       extreme eigenvectors of $\mathbf A^\top\mathbf A$.
    1. Divide every column of $\mathbf{A}$ by its norm and repeat the
       measurement. How does the minimizer of the rescaled problem map back
       to $\mathbf{w}^\star$?
    1. Part 2 of Exercise
       1 of
       that section writes gradient descent
       in new coordinates $\mathbf{u}$ with $\mathbf{w} = \mathbf T\mathbf{u}$
       as gradient descent on the original problem preconditioned by a
       positive definite matrix $\mathbf P$, with update
       $\mathbf{w} \leftarrow \mathbf{w} - \eta\,\mathbf{P}^{-1}\nabla f(\mathbf{w})$.
       Which $\mathbf T$ and which $\mathbf P$ correspond to column
       scaling? How is $\mathbf P$ related to the diagonal of
       $\mathbf{A}^\top\mathbf{A}$?

1. [code] **Stochastic rounding.** ● Stochastic rounding
   [@Gupta.Agrawal.Gopalakrishnan.ea.2015] rounds a real number $x$
   between adjacent representable values $x_- < x_+$ to $x_+$ with
   probability $q = (x - x_-)/(x_+ - x_-)$ and to $x_-$ otherwise.
    1. Show that the rounded value has expectation $x$ and variance
       $q(1 - q)\Delta^2 \le \Delta^2/4$, where $\Delta = x_+ - x_-$.
    1. A bf16 weight starts at $w = 1$ and receives $K = 1000$ updates
       of $-10^{-4}$, each followed by stochastic rounding. The weight stays
       in $[0.5, 1]$, where the bf16 spacing is $\Delta = 2^{-8}$. Show
       that the rounding errors of different steps are uncorrelated, and
       predict the mean and the standard deviation of the final weight.
       Which final weight does round-to-nearest produce?
    1. For a positive fp32 number, adding a uniformly random integer
       from $\{0, \ldots, 2^{16} - 1\}$ to its bit pattern and then clearing
       the lower 16 bits rounds it stochastically to bf16. Explain why.
       Implement this rounding next to the `to_bf16` function of
       that section and check the predictions of the
       previous sub-item with 1000 independent runs. Compare with an fp32
       master copy that receives the same updates.

1. **Exact subtraction.** ● Subtraction can be exact and still return an
   inaccurate difference.
    1. Show that the bound the equation is
       attained for suitable $\delta_a$ and $\delta_b$ with
       $|\delta_a|,|\delta_b|\le u$.
    1. Prove Sterbenz's lemma [@Sterbenz.1974]: if $a$ and $b$ are
       floating-point numbers in the format
       the equation with $b/2 \le a \le 2b$, then
       $a - b$ is a floating-point number, so the computed difference is
       exact. Assume that the format includes subnormal numbers.
    1. In the `#numerical-stability-conditioning-log1p` cell, the fp32
       difference $a - b$ misses the true value $3.0 \times 10^{-7}$ by
       about 20 percent. Use Sterbenz's lemma and the printed number of
       ulps to verify that the subtraction was exact, compute the relative
       errors of the stored $a$ and $b$, and check that the bound of the
       first sub-item covers the error.

1. [code] **Merging running moments.** ● Data-parallel training computes
   statistics on shards and combines them. Let shard $A$ hold $k_A$ values
   with mean $m_A$ and centered sum of squares $M_A$, and let shard $B$
   hold $k_B$ values with $m_B$ and $M_B$.
    1. Derive exact formulas for the mean $m$ and the centered sum of
       squares $M$ of the union of the two shards in terms of $k_A$, $m_A$,
       $M_A$, $k_B$, $m_B$, and $M_B$.
    1. Show that the case $k_B = 1$ is Welford's
       recursion the equation.
    1. Split the $10^5$ samples of the
       `#numerical-stability-conditioning-welford` cell into $2^j$ shards
       of nearly equal size (`np.array_split`) for $j \in \{1, 4, 8\}$.
       Compute each shard's statistics with
       Welford's recursion, merge them pairwise in a binary tree, and
       compare the variance with the two-pass reference. Merge the shards a
       second time by adding their sums $\sum_i x_i$ and $\sum_i x_i^2$ and
       applying the one-pass formula. Explain the difference.

1. [code] **Summation error growth.** ● Let $S_n = \sum_{i=1}^n x_i$ and
   let $\hat S_n$ be its computed value. In-order summation has the
   worst-case bound $|\hat S_n - S_n| \le (n - 1)\,u \sum_i |x_i| + O(u^2)$.
    1. For $n = 2^k$, recursive pairwise summation splits the terms into
       two halves, sums each half recursively, and adds the two results.
       Prove that it satisfies
       $|\hat S_n - S_n| \le u \log_2(n) \sum_i |x_i| + O(u^2)$.
    1. For $n \in \{2^4, 2^8, 2^{12}, 2^{16}, 2^{20}\}$ uniform random
       numbers in $[0, 1)$ stored as fp32, measure the relative errors of
       in-order summation (`np.cumsum(x)[-1]`), of `np.sum` (a blocked
       pairwise summation), and of Kahan's compensated summation
       [@Kahan.1965] against a float64 reference. Kahan's method keeps
       a correction $c$: after each addition $t = s + y$ it sets
       $c = (t - s) - y$ and subtracts $c$ from the next term. Plot the
       three errors against $n$ together with the two bounds.
    1. How does the measured error of in-order summation grow with $n$
       compared with its worst-case bound, and why? Why is the text's sum
       of $10^7$ copies of $0.1$ so much closer to the worst case?
    1. What limits the error of the two accurate methods at large $n$?

    *Adapted from Steven G. Johnson, [MIT 18.335 Introduction to Numerical Methods, Problem Set 1](https://ocw.mit.edu/courses/18-335j-introduction-to-numerical-methods-spring-2019/6e6730885b20bd046cbde805d15a4835_MIT18_335JS19_pset1.pdf), Problem 4.*

1. [code] **Mixed-precision iterative refinement.** ●● Mixed-precision
   training does the bulk of its arithmetic in low precision and keeps the
   quantities that accumulate small corrections in high precision. Iterative
   refinement [@Moler.1967; @Carson.Higham.2018] applies the same split
   to $\mathbf{A}\mathbf{x} = \mathbf{b}$: the $O(n^3)$ factorization runs
   in low precision and only $O(n^2)$ residual work runs in high precision.
   Starting from an fp32 solution $\mathbf{x}_0$, it repeats
   $$
   \mathbf{r}_k = \mathbf{b} - \mathbf{A}\mathbf{x}_k, \qquad
   (\mathbf{A} + \Delta\mathbf{A}_k)\,\mathbf{d}_k = \mathbf{r}_k, \qquad
   \mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{d}_k,
   $$
   where the residual and the update are computed in fp64 and the
   correction $\mathbf{d}_k$ comes from an fp32 solve.
    1. Model the fp32 solve as backward stable,
       $\|\Delta\mathbf{A}_k\| \le c\,u_{32}\,\|\mathbf{A}\|$ with
       $u_{32} = 2^{-24}$ and a moderate constant $c$, and neglect the
       fp64 rounding errors. Show that the error
       $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}$ satisfies
       $\mathbf{e}_{k+1} = (\mathbf{A} + \Delta\mathbf{A}_k)^{-1}
       \Delta\mathbf{A}_k\,\mathbf{e}_k$. Derive a contraction factor in
       terms of $c\,\kappa(\mathbf{A})\,u_{32}$, and state when it is below
       $1$.
    1. Implement the iteration for the Hilbert matrices of the
       `#numerical-stability-conditioning-hilbert` cell with
       $n = 3, \ldots, 10$, using `np.linalg.solve` on fp32 copies of
       $\mathbf{H}$ and $\mathbf{r}_k$ for the correction. Record the
       forward error over ten iterations and compare its final value with
       that of an fp64 solve. For which $n$ does refinement reach the
       accuracy of the fp64 solve? For which $n$ does it still converge,
       but too slowly to reach that accuracy within ten iterations? For
       which $n$ does the error grow?
    1. Estimate the measured contraction per iteration for each $n$ and
       compare it with $\kappa(\mathbf{H})\,u_{32}$ and with the spectral
       radius of $\mathbf{I} - \hat{\mathbf{H}}^{-1}\mathbf{H}$, where
       $\hat{\mathbf{H}}$ is the fp32 copy of $\mathbf{H}$ and the
       inverse is applied in fp64. Which perturbation dominates
       $\Delta\mathbf{A}_k$ in this experiment, and how pessimistic is the
       bound of the first sub-item?

[Discussions](https://d2l.discourse.group/t/36832)